# Evaluation: Data Preprocessing Node
This notebook evaluates `data_preprocessing` predictions against benchmark labels and keeps iterative re-runs tidy with a reusable function.

In [1]:
# Setup imports and project paths
import os
import sys
import json
from pathlib import Path

import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

# Resolve project root robustly from notebook location
ROOT = Path.cwd().resolve()
if not (ROOT / "nodes").exists():
    ROOT = ROOT.parent
if not (ROOT / "nodes").exists():
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

PILOT = ROOT / "Pilot_Evaluation"
BENCHMARK_FILE = PILOT / "DATA_sample_10" / "Data Science Research Process (DSRP) Framework.csv"
RESULTS_FOLDER = PILOT / "pilot_study_results"
VECTOR_DB_PATH = PILOT / "pilot_study_10_vectors"
COLLECTION_NAME = "pilot_study_10"
EMBEDDING_MODEL = "text-embedding-3-small"

print(f"Project root: {ROOT}")
print(f"Benchmark file exists: {BENCHMARK_FILE.exists()}")
print(f"Results folder exists: {RESULTS_FOLDER.exists()}")

Project root: C:\Users\sahil\OneDrive\PhD\3. Empirical Study\Methodological Workflow
Benchmark file exists: True
Results folder exists: True


In [2]:
# Load benchmark and define preprocessing helpers
DIMENSIONS = ["Data Cleaning", "Data Reduction", "Data Transformation"]
PRED_KEY_MAP = {
    "Data Cleaning": "validated_data_cleaning",
    "Data Reduction": "validated_data_reduction",
    "Data Transformation": "validated_data_transformation",
}

CANONICAL_STATUSES = ["Not Reported", "Mentioned", "Transparently Described"]
STATUS_MAP = {
    "not reported": "Not Reported",
    "mentioned": "Mentioned",
    "transparently described": "Transparently Described",
}

def normalize_text(value):
    if value is None or pd.isna(value):
        return None
    text = str(value).strip()
    return text if text else None

def normalize_paper_id(value):
    text = normalize_text(value)
    return text.lower() if text else None

def normalize_status(value):
    text = normalize_text(value)
    if not text:
        return "Not Reported"
    return STATUS_MAP.get(text.lower(), text)

def get_nested(data, path, default=None):
    current = data
    for key in path.split("."):
        if not isinstance(current, dict) or key not in current:
            return default
        current = current[key]
    return current

def extract_predicted_status(dp_obj, benchmark_dim):
    pred_key = PRED_KEY_MAP[benchmark_dim]
    raw_value = dp_obj.get(pred_key, {})
    if isinstance(raw_value, dict):
        raw_value = raw_value.get("status")
    return normalize_status(raw_value)

benchmark_df = pd.read_csv(BENCHMARK_FILE)
benchmark_df.columns = benchmark_df.columns.str.strip()

required_cols = ["Paper ID", "Gatekeeper", *DIMENSIONS]
missing = [c for c in required_cols if c not in benchmark_df.columns]
if missing:
    raise ValueError(f"Missing required benchmark columns: {missing}")

benchmark_eval = benchmark_df[required_cols].copy()
benchmark_eval["Paper ID"] = benchmark_eval["Paper ID"].apply(normalize_paper_id)
benchmark_eval["Gatekeeper"] = benchmark_eval["Gatekeeper"].astype(str).str.strip().str.lower()
for dim in DIMENSIONS:
    benchmark_eval[dim] = benchmark_eval[dim].apply(normalize_status)

benchmark_eval = benchmark_eval[benchmark_eval["Gatekeeper"] == "include"].copy()
benchmark_eval = benchmark_eval.dropna(subset=["Paper ID"]).reset_index(drop=True)

print(f"Included benchmark rows: {len(benchmark_eval)}")
print(benchmark_eval[["Paper ID", *DIMENSIONS]].head().to_string(index=False))

Included benchmark rows: 7
 Paper ID           Data Cleaning Data Reduction     Data Transformation
 2018 - 8            Not Reported   Not Reported            Not Reported
2019 - 33            Not Reported   Not Reported            Not Reported
 2020 - 8 Transparently Described   Not Reported Transparently Described
2021 - 28 Transparently Described   Not Reported Transparently Described
2021 - 56               Mentioned   Not Reported Transparently Described


In [3]:
# Load existing agent outputs and build comparison table
def load_agent_results(results_folder: Path):
    agent_data = {}
    for paper_dir in sorted(results_folder.glob("*/")):
        if not paper_dir.is_dir():
            continue
        results_file = paper_dir / "aggregated" / "results.json"
        if not results_file.exists():
            continue
        with open(results_file, "r", encoding="utf-8") as f:
            agent_data[paper_dir.name.strip().lower()] = json.load(f)
    return agent_data

agent_data = load_agent_results(RESULTS_FOLDER)
print(f"Loaded aggregated outputs for {len(agent_data)} papers")

comparison_rows = []
missing_agent_results = []

for _, row in benchmark_eval.iterrows():
    paper_id = row["Paper ID"]
    if paper_id not in agent_data:
        missing_agent_results.append(paper_id)
        continue

    outputs = agent_data[paper_id]
    dp_obj = get_nested(outputs, "dsrp_outputs.data_preprocessing", default={})

    record = {"Paper ID": paper_id}
    dim_match_flags = []
    for dim in DIMENSIONS:
        gt_value = normalize_status(row[dim])
        pred_value = extract_predicted_status(dp_obj, dim)
        is_match = gt_value == pred_value
        dim_match_flags.append(is_match)

        record[f"GT_{dim}"] = gt_value
        record[f"Agent_{dim}"] = pred_value
        record[f"Match_{dim}"] = "MATCH" if is_match else "MISMATCH"

    record["Match_All_3"] = "MATCH" if all(dim_match_flags) else "MISMATCH"
    comparison_rows.append(record)

comparison_df = pd.DataFrame(comparison_rows)
print(f"Comparable rows: {len(comparison_df)}")
if missing_agent_results:
    print(f"Missing agent outputs for {len(missing_agent_results)} papers: {', '.join(missing_agent_results)}")

if len(comparison_df):
    show_cols = ["Paper ID", *[f"Match_{d}" for d in DIMENSIONS], "Match_All_3"]
    print(comparison_df[show_cols].to_string(index=False))
else:
    print("No comparable rows found.")

Loaded aggregated outputs for 10 papers
Comparable rows: 7
 Paper ID Match_Data Cleaning Match_Data Reduction Match_Data Transformation Match_All_3
 2018 - 8               MATCH             MISMATCH                     MATCH    MISMATCH
2019 - 33               MATCH             MISMATCH                     MATCH    MISMATCH
 2020 - 8               MATCH                MATCH                  MISMATCH    MISMATCH
2021 - 28               MATCH                MATCH                     MATCH       MATCH
2021 - 56            MISMATCH             MISMATCH                     MATCH    MISMATCH
2023 - 58               MATCH             MISMATCH                     MATCH    MISMATCH
2024 - 99               MATCH                MATCH                  MISMATCH    MISMATCH


In [4]:
# Single-label metrics per preprocessing dimension
if len(comparison_df) == 0:
    raise ValueError("No rows available for evaluation.")

metrics_rows = []
for dim in DIMENSIONS:
    y_true = comparison_df[f"GT_{dim}"].tolist()
    y_pred = comparison_df[f"Agent_{dim}"].tolist()

    acc = accuracy_score(y_true, y_pred)
    precision_macro = precision_score(y_true, y_pred, labels=CANONICAL_STATUSES, average="macro", zero_division=0)
    recall_macro = recall_score(y_true, y_pred, labels=CANONICAL_STATUSES, average="macro", zero_division=0)
    f1_macro = f1_score(y_true, y_pred, labels=CANONICAL_STATUSES, average="macro", zero_division=0)

    metrics_rows.append({
        "dimension": dim,
        "samples": len(y_true),
        "accuracy": acc,
        "precision_macro": precision_macro,
        "recall_macro": recall_macro,
        "f1_macro": f1_macro,
    })

    print("=" * 90)
    print(f"DIMENSION: {dim}")
    print("=" * 90)
    print(f"accuracy............ {acc:.4f}")
    print(f"precision_macro..... {precision_macro:.4f}")
    print(f"recall_macro........ {recall_macro:.4f}")
    print(f"f1_macro............ {f1_macro:.4f}")

    report = classification_report(
        y_true,
        y_pred,
        labels=CANONICAL_STATUSES,
        target_names=CANONICAL_STATUSES,
        zero_division=0,
        output_dict=True,
    )
    report_df = pd.DataFrame(report).T
    print("\nPer-label report:")
    print(report_df.to_string())

metrics_df = pd.DataFrame(metrics_rows)
overall_match_count = int((comparison_df["Match_All_3"] == "MATCH").sum())
overall_total = len(comparison_df)
print("\nSUMMARY TABLE")
print(metrics_df.to_string(index=False))
print(f"\nOverall exact agreement across all 3 preprocessing dimensions: {overall_match_count}/{overall_total} ({100 * overall_match_count / overall_total:.1f}%)")

DIMENSION: Data Cleaning
accuracy............ 0.8571
precision_macro..... 0.6000
recall_macro........ 0.6667
f1_macro............ 0.6296

Per-label report:
                         precision    recall  f1-score   support
Not Reported              0.800000  1.000000  0.888889  4.000000
Mentioned                 0.000000  0.000000  0.000000  1.000000
Transparently Described   1.000000  1.000000  1.000000  2.000000
accuracy                  0.857143  0.857143  0.857143  0.857143
macro avg                 0.600000  0.666667  0.629630  7.000000
weighted avg              0.742857  0.857143  0.793651  7.000000
DIMENSION: Data Reduction
accuracy............ 0.4286
precision_macro..... 0.3333
recall_macro........ 0.1429
f1_macro............ 0.2000

Per-label report:
                         precision    recall  f1-score   support
Not Reported              1.000000  0.428571  0.600000  7.000000
Mentioned                 0.000000  0.000000  0.000000  0.000000
Transparently Described   0.000000  0

In [5]:
# Detailed mismatch diagnostics
detailed = comparison_df.copy()
for dim in DIMENSIONS:
    detailed[f"Pair_{dim}"] = detailed.apply(
        lambda r: f"GT={r[f'GT_{dim}']} | Pred={r[f'Agent_{dim}']}",
        axis=1,
    )

mismatch_df = detailed[detailed["Match_All_3"] == "MISMATCH"].copy()
print(f"Total mismatched papers: {len(mismatch_df)}")
if len(mismatch_df):
    print(mismatch_df[["Paper ID", *[f"Pair_{d}" for d in DIMENSIONS], "Match_All_3"]].to_string(index=False))
else:
    print("No mismatches found.")

Total mismatched papers: 6
 Paper ID                                        Pair_Data Cleaning                            Pair_Data Reduction                                  Pair_Data Transformation Match_All_3
 2018 - 8                       GT=Not Reported | Pred=Not Reported GT=Not Reported | Pred=Transparently Described                       GT=Not Reported | Pred=Not Reported    MISMATCH
2019 - 33                       GT=Not Reported | Pred=Not Reported               GT=Not Reported | Pred=Mentioned                       GT=Not Reported | Pred=Not Reported    MISMATCH
 2020 - 8 GT=Transparently Described | Pred=Transparently Described            GT=Not Reported | Pred=Not Reported            GT=Transparently Described | Pred=Not Reported    MISMATCH
2021 - 56                          GT=Mentioned | Pred=Not Reported GT=Not Reported | Pred=Transparently Described GT=Transparently Described | Pred=Transparently Described    MISMATCH
2023 - 58                       GT=Not Reported 

## Iteration Utility
Use the function below to re-run `data_preprocessing_node` on any paper list while keeping iteration code out of your main analysis cells.

In [6]:
import importlib
import textwrap
import re
import nodes.data_preprocessing_node as data_preprocessing_module
from utils.dsrp_state import DSRPState

def _extract_dimension_justification_from_reasoning(validated_reasoning, dimension):
    """
    Extract justification for a specific dimension from the overall validated_reasoning.
    Looks for paragraphs or sentences that reference the dimension.
    """
    if not validated_reasoning:
        return ""
    
    # Try to find sentences or clauses mentioning the dimension
    dim_short = dimension.split()[-1].lower()  # "Cleaning", "Reduction", "Transformation"
    
    # Look for patterns like "Data Cleaning is ...", "Data Reduction stated ...", etc.
    patterns = [
        f"{dimension}.*?(?:\\.|;|$)",  # Match from dimension name until period/semicolon/EOL
        f"{dim_short}.*?(?:\\.|;|$)",  # Match from short name
    ]
    
    for pattern in patterns:
        matches = re.findall(pattern, validated_reasoning, re.IGNORECASE | re.DOTALL)
        if matches:
            # Return the first match, cleaned up
            justif = matches[0].strip()
            if len(justif) > 10:  # Reasonable length check
                return justif
    
    return ""

def _format_bibliography_text(bibliography):
    entries = bibliography or []
    if not entries:
        return ["(no bibliography)"]
    lines = []
    for i, b in enumerate(entries, start=1):
        sec = b.get("section", "")
        page = b.get("page", "")
        quote = b.get("direct_quote", "")
        lines.append(f"[{i}] page={page} | section={sec}")
        if quote:
            wrapped = textwrap.wrap(str(quote), width=110)
            if wrapped:
                lines.extend([f"    quote: {wrapped[0]}"] + [f"           {w}" for w in wrapped[1:]])
    return lines

def _extract_reasoning_payload(dp_obj):
    payload = {
        "overall_validated_reasoning": dp_obj.get("validated_reasoning", ""),
        "audit_commentary": dp_obj.get("audit_commentary", ""),
        "confidence": dp_obj.get("confidence", None),
        "validated_bibliography_count": len(dp_obj.get("validated_bibliography", []) or []),
        "validated_bibliography_lines": _format_bibliography_text(dp_obj.get("validated_bibliography", [])),
    }

    for dim in DIMENSIONS:
        key = PRED_KEY_MAP[dim]
        raw_obj = dp_obj.get(key, {})
        if not isinstance(raw_obj, dict):
            raw_obj = {}
        
        # Try to get justification from the raw object first
        justification = raw_obj.get("justification", "")
        
        # If empty, try to extract from validated_reasoning
        if not justification:
            justification = _extract_dimension_justification_from_reasoning(
                payload["overall_validated_reasoning"],
                dim
            )
        
        payload[f"Justification_{dim}"] = justification
        payload[f"Raw_{dim}_obj"] = json.dumps(raw_obj, ensure_ascii=False, indent=2)
    return payload

def _print_label_block(record, dim, show_raw=False, show_bibliography=False):
    print(f"\n{dim}")
    print(f"  GT    : {record.get(f'GT_{dim}', '(missing)')}")
    print(f"  Agent : {record.get(f'New_Agent_{dim}', '(missing)')}")
    print(f"  Match : {record.get(f'Match_{dim}', '(missing)')}")
    print("  Justification:")
    just = str(record.get(f"Justification_{dim}", "") or "(none)")
    wrapped_just = textwrap.wrap(just, width=105)
    if wrapped_just:
        for line in wrapped_just:
            print(f"    {line}")
    else:
        print("    (none)")
    if show_raw:
        print("  Raw label output:")
        print(textwrap.indent(str(record.get(f"Raw_{dim}_obj", "{}")), "    "))
    if show_bibliography:
        print("  Bibliography:")
        for line in record.get("validated_bibliography_lines", ["(no bibliography)"]):
            print(f"    {line}")

def _print_iteration_pretty(record, show_raw=False, show_bibliography=True):
    print("\n" + "=" * 100)
    print(f"PAPER: {record.get('Paper ID', '(missing)')} | MODEL: {record.get('Model', '(missing)')}")
    print("=" * 100)
    if record.get("Error"):
        print(f"Error: {record['Error']}")
        return

    for dim in DIMENSIONS:
        _print_label_block(record, dim, show_raw=show_raw, show_bibliography=show_bibliography)

    print("\nOverall all-3-dimensions match:", record.get("Match_All_3", "(missing)"))
    print("Confidence:", record.get("confidence", "(missing)"))
    print("Overall validated reasoning:")
    reasoning = str(record.get("overall_validated_reasoning", "") or "(none)")
    for line in textwrap.wrap(reasoning, width=110):
        print(f"  {line}")
    print("Audit commentary:")
    commentary = str(record.get("audit_commentary", "") or "(none)")
    for line in textwrap.wrap(commentary, width=110):
        print(f"  {line}")

def run_data_preprocessing_iteration(
    paper_ids,
    llm_model="gpt-4o-mini",
    verbose=True,
    include_reasoning=True,
    show_labels_in_logs=True,
    show_reasoning_in_logs=False,
    log_bibliography=False,
    show_raw_label_output=False,
    ):
    """Run data_preprocessing_node over papers; keep batch output concise while still showing assigned labels."""
    importlib.reload(data_preprocessing_module)
    data_preprocessing_node = data_preprocessing_module.data_preprocessing_node

    benchmark_lookup = benchmark_eval.set_index("Paper ID")
    rows = []

    original_dir = Path.cwd().resolve()
    os.chdir(ROOT)
    try:
        for paper_id in paper_ids:
            pid = normalize_paper_id(paper_id)
            if not pid:
                continue
            if pid not in benchmark_lookup.index:
                rows.append({"Paper ID": pid, "Error": "Paper ID not found in benchmark include set"})
                continue

            print(f"Re-running data_preprocessing node for {pid} with {llm_model}...")
            state = DSRPState(
                paper_id=pid,
                gatekeeper={},
                dsrp_outputs={},
                collection_name=COLLECTION_NAME,
                persist_directory=str(VECTOR_DB_PATH),
                embedding_model=EMBEDDING_MODEL,
                llm_model=llm_model,
            )

            try:
                out = data_preprocessing_node(state)
                dp_obj = get_nested(out, "dsrp_outputs.data_preprocessing", default={})

                record = {"Paper ID": pid, "Model": llm_model}
                match_flags = []
                for dim in DIMENSIONS:
                    gt_value = normalize_status(benchmark_lookup.loc[pid, dim])
                    pred_value = extract_predicted_status(dp_obj, dim)
                    dim_match = gt_value == pred_value
                    match_flags.append(dim_match)

                    record[f"GT_{dim}"] = gt_value
                    record[f"New_Agent_{dim}"] = pred_value
                    record[f"Match_{dim}"] = "MATCH" if dim_match else "MISMATCH"

                if include_reasoning:
                    record.update(_extract_reasoning_payload(dp_obj))

                record["Match_All_3"] = "MATCH" if all(match_flags) else "MISMATCH"
                rows.append(record)

                if verbose and show_labels_in_logs:
                    label_line = " | ".join(
                        f"{dim}: {record.get(f'New_Agent_{dim}', '(missing)')}"
                        for dim in DIMENSIONS
                    )
                    print(f"  Assigned labels -> {label_line}")
                    print(f"  Match_All_3: {record['Match_All_3']}")

                if verbose and include_reasoning and show_reasoning_in_logs:
                    _print_iteration_pretty(
                        record,
                        show_raw=show_raw_label_output,
                        show_bibliography=log_bibliography,
                    )

            except Exception as e:
                err_row = {"Paper ID": pid, "Model": llm_model, "Error": str(e)}
                rows.append(err_row)
                if verbose:
                    print(f"  Error: {e}")
    finally:
        os.chdir(original_dir)

    retest_df = pd.DataFrame(rows)
    if len(retest_df):
        print("\n" + "-" * 100)
        print("RE-TEST RESULTS (compact summary)")
        print("-" * 100)
        show_cols = ["Paper ID", "Model", *[col for d in DIMENSIONS for col in (f"GT_{d}", f"New_Agent_{d}", f"Match_{d}")], "Match_All_3"]
        cols = [c for c in show_cols if c in retest_df.columns]
        if cols:
            print(retest_df[cols].to_string(index=False))
        else:
            print(retest_df.to_string(index=False))
    return retest_df

def show_iteration_reasoning(retest_df, paper_id, show_raw_label_output=False, show_bibliography=True):
    pid = normalize_paper_id(paper_id)
    if "Paper ID" not in retest_df.columns:
        raise ValueError("retest_df does not contain 'Paper ID'.")

    rows = retest_df[retest_df["Paper ID"] == pid]
    if len(rows) == 0:
        raise ValueError(f"Paper ID not found in retest_df: {pid}")

    row = rows.iloc[0].to_dict()
    _print_iteration_pretty(
        row,
        show_raw=show_raw_label_output,
        show_bibliography=show_bibliography,
    )

def watch_reasoning_post_finalization(
    retest_df,
    paper_ids=None,
    only_mismatches=False,
    max_papers=None,
    show_raw_label_output=False,
    show_bibliography=True,
    ):
    """Inspect detailed reasoning after a batch run without cluttering the main run output."""
    if "Paper ID" not in retest_df.columns:
        raise ValueError("retest_df does not contain 'Paper ID'.")

    view_df = retest_df.copy()
    if only_mismatches and "Match_All_3" in view_df.columns:
        view_df = view_df[view_df["Match_All_3"] != "MATCH"]

    if paper_ids is not None:
        wanted = {normalize_paper_id(p) for p in paper_ids if normalize_paper_id(p)}
        view_df = view_df[view_df["Paper ID"].isin(wanted)]

    if max_papers is not None:
        view_df = view_df.head(int(max_papers))

    if len(view_df) == 0:
        print("No papers found for reasoning view with current filters.")
        return

    print(f"Showing reasoning for {len(view_df)} paper(s).")
    for pid in view_df["Paper ID"].tolist():
        show_iteration_reasoning(
            retest_df,
            pid,
            show_raw_label_output=show_raw_label_output,
            show_bibliography=show_bibliography,
        )

In [7]:
# Example usage (kept short to avoid clutter)
papers_to_rerun = comparison_df[comparison_df["Match_All_3"] == "MISMATCH"]["Paper ID"].tolist()
print(f"Papers selected for re-run: {len(papers_to_rerun)}")

# Uncomment one of the calls below when you want to run iterations with reasoning included.
# Full mismatch re-run:
# retest_df = run_data_preprocessing_iteration(
#     paper_ids=papers_to_rerun,
#     llm_model="gpt-4o-mini",
#     verbose=True,
#     include_reasoning=True,
#     show_reasoning_in_logs=False,
#     log_bibliography=False,
# )

# Quick smoke test for one paper:
# retest_df = run_data_preprocessing_iteration(
#     ["2018 - 12"],
#     llm_model="gpt-4o-mini",
#     include_reasoning=True,
#     show_reasoning_in_logs=True,
#     log_bibliography=True,
# )

Papers selected for re-run: 6


## Reasoning Debug View
After running `run_data_preprocessing_iteration(...)`, use this to inspect one paper's label-by-label reasoning and bibliography.

# **Iteration Cells Below**

### **Re-run on ALL Papers**

In [8]:
papers_to_rerun = ['2018 - 8', '2019 - 33', '2020 - 8', '2021 - 28', '2021 - 56', '2023 - 58', '2024 - 99']

**Iteration 1** No changes just experimenting with current **LLM Temp 0.5**

In [32]:
# Batch run (quiet reasoning, but show assigned labels per paper)
retest_df = run_data_preprocessing_iteration(
    papers_to_rerun,
    llm_model="gpt-4o-mini",
    verbose=True,
    include_reasoning=True,
    show_labels_in_logs=True,
    show_reasoning_in_logs=False,
    log_bibliography=False,
    show_raw_label_output=False,
 )

Re-running data_preprocessing node for 2018 - 8 with gpt-4o-mini...
  Assigned labels -> Data Cleaning: Not Reported | Data Reduction: Transparently Described | Data Transformation: Not Reported
  Match_All_3: MISMATCH
Re-running data_preprocessing node for 2019 - 33 with gpt-4o-mini...
  Assigned labels -> Data Cleaning: Not Reported | Data Reduction: Mentioned | Data Transformation: Not Reported
  Match_All_3: MISMATCH
Re-running data_preprocessing node for 2020 - 8 with gpt-4o-mini...
  Assigned labels -> Data Cleaning: Transparently Described | Data Reduction: Not Reported | Data Transformation: Not Reported
  Match_All_3: MISMATCH
Re-running data_preprocessing node for 2021 - 28 with gpt-4o-mini...
  Assigned labels -> Data Cleaning: Transparently Described | Data Reduction: Transparently Described | Data Transformation: Transparently Described
  Match_All_3: MATCH
Re-running data_preprocessing node for 2021 - 56 with gpt-4o-mini...
  Assigned labels -> Data Cleaning: Mentioned | 

**Iteration 2** After major prompt refinement. 

In [13]:
# Batch run (quiet reasoning, but show assigned labels per paper)
retest_df = run_data_preprocessing_iteration(
    papers_to_rerun,
    llm_model="gpt-4o-mini",
    verbose=True,
    include_reasoning=True,
    show_labels_in_logs=False,
    show_reasoning_in_logs=False,
    log_bibliography=False,
    show_raw_label_output=False,
 )

Re-running data_preprocessing node for 2018 - 8 with gpt-4o-mini...
Re-running data_preprocessing node for 2019 - 33 with gpt-4o-mini...
Re-running data_preprocessing node for 2020 - 8 with gpt-4o-mini...
Re-running data_preprocessing node for 2021 - 28 with gpt-4o-mini...
Re-running data_preprocessing node for 2021 - 56 with gpt-4o-mini...
Re-running data_preprocessing node for 2023 - 58 with gpt-4o-mini...
Re-running data_preprocessing node for 2024 - 99 with gpt-4o-mini...

----------------------------------------------------------------------------------------------------
RE-TEST RESULTS (compact summary)
----------------------------------------------------------------------------------------------------
 Paper ID       Model New_Agent_Data Cleaning New_Agent_Data Reduction New_Agent_Data Transformation Match_All_3
 2018 - 8 gpt-4o-mini            Not Reported             Not Reported                  Not Reported    MISMATCH
2019 - 33 gpt-4o-mini            Not Reported           

In [26]:
# Compact view with GT + New Agent + per-dimension Match labels (no rerun needed if retest_df already exists)
if "retest_df" in globals() and len(retest_df):
    show_cols = ["Paper ID", "Model", *[col for d in DIMENSIONS for col in (f"GT_{d}", f"New_Agent_{d}", f"Match_{d}")], "Match_All_3"]
    cols = [c for c in show_cols if c in retest_df.columns]
    print(retest_df[cols].to_string(index=False) if cols else retest_df.to_string(index=False))
else:
    print("retest_df not found. Run the batch re-test cell first.")

 Paper ID       Model        GT_Data Cleaning New_Agent_Data Cleaning Match_Data Cleaning       GT_Data Reduction New_Agent_Data Reduction Match_Data Reduction  GT_Data Transformation New_Agent_Data Transformation Match_Data Transformation Match_All_3
 2018 - 8 gpt-4o-mini            Not Reported            Not Reported               MATCH Transparently Described             Not Reported             MISMATCH               Mentioned                  Not Reported                  MISMATCH    MISMATCH
2019 - 33 gpt-4o-mini            Not Reported            Not Reported               MATCH            Not Reported                Mentioned             MISMATCH            Not Reported                  Not Reported                     MATCH    MISMATCH
 2020 - 8 gpt-4o-mini               Mentioned Transparently Described            MISMATCH               Mentioned             Not Reported             MISMATCH Transparently Described                  Not Reported                  MISMATCH    M

**Iteration 3** After fixing prompts based on first two studies, minor adjustment in data reduction. 

In [46]:
# Batch run (quiet reasoning, but show assigned labels per paper)
retest_df = run_data_preprocessing_iteration(
    papers_to_rerun,
    llm_model="gpt-4o-mini",
    verbose=True,
    include_reasoning=True,
    show_labels_in_logs=False,
    show_reasoning_in_logs=False,
    log_bibliography=False,
    show_raw_label_output=False,
 )

Re-running data_preprocessing node for 2018 - 8 with gpt-4o-mini...
Re-running data_preprocessing node for 2019 - 33 with gpt-4o-mini...
Re-running data_preprocessing node for 2020 - 8 with gpt-4o-mini...
Re-running data_preprocessing node for 2021 - 28 with gpt-4o-mini...
Re-running data_preprocessing node for 2021 - 56 with gpt-4o-mini...
Re-running data_preprocessing node for 2023 - 58 with gpt-4o-mini...
Re-running data_preprocessing node for 2024 - 99 with gpt-4o-mini...

----------------------------------------------------------------------------------------------------
RE-TEST RESULTS (compact summary)
----------------------------------------------------------------------------------------------------
 Paper ID       Model        GT_Data Cleaning New_Agent_Data Cleaning Match_Data Cleaning       GT_Data Reduction New_Agent_Data Reduction Match_Data Reduction  GT_Data Transformation New_Agent_Data Transformation Match_Data Transformation Match_All_3
 2018 - 8 gpt-4o-mini         

**Iteration 4** After fixing based on Study 3 i.e., 2020 - 8, minor rule added to label text preprocessing such as ngrams formation as trnasparently described. 

In [ ]:
# Batch run (quiet reasoning, but show assigned labels per paper)
retest_df = run_data_preprocessing_iteration(
    papers_to_rerun,
    llm_model="gpt-4o-mini",
    verbose=True,
    include_reasoning=True,
    show_labels_in_logs=False,
    show_reasoning_in_logs=False,
    log_bibliography=False,
    show_raw_label_output=True,
 )

Re-test.

In [72]:
# Batch run (quiet reasoning, but show assigned labels per paper)
retest_df = run_data_preprocessing_iteration(
    papers_to_rerun,
    llm_model="gpt-4o-mini",
    verbose=True,
    include_reasoning=True,
    show_labels_in_logs=False,
    show_reasoning_in_logs=False,
    log_bibliography=False,
    show_raw_label_output=True,
 )

Re-running data_preprocessing node for 2018 - 8 with gpt-4o-mini...
Re-running data_preprocessing node for 2019 - 33 with gpt-4o-mini...
Re-running data_preprocessing node for 2020 - 8 with gpt-4o-mini...
Re-running data_preprocessing node for 2021 - 28 with gpt-4o-mini...
Re-running data_preprocessing node for 2021 - 56 with gpt-4o-mini...
Re-running data_preprocessing node for 2023 - 58 with gpt-4o-mini...
Re-running data_preprocessing node for 2024 - 99 with gpt-4o-mini...

----------------------------------------------------------------------------------------------------
RE-TEST RESULTS (compact summary)
----------------------------------------------------------------------------------------------------
 Paper ID       Model        GT_Data Cleaning New_Agent_Data Cleaning Match_Data Cleaning       GT_Data Reduction New_Agent_Data Reduction Match_Data Reduction  GT_Data Transformation New_Agent_Data Transformation Match_Data Transformation Match_All_3
 2018 - 8 gpt-4o-mini         

In [11]:
# Batch run (quiet reasoning, but show assigned labels per paper)
retest_df = run_data_preprocessing_iteration(
    papers_to_rerun,
    llm_model="gpt-4o-mini",
    verbose=True,
    include_reasoning=True,
    show_labels_in_logs=False,
    show_reasoning_in_logs=False,
    log_bibliography=False,
    show_raw_label_output=True,
 )

Re-running data_preprocessing node for 2018 - 8 with gpt-4o-mini...


C:\Users\sahil\OneDrive\PhD\3. Empirical Study\Methodological Workflow\utils\paper_retriever.py:9: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  self._vectorstore = Chroma(


Re-running data_preprocessing node for 2019 - 33 with gpt-4o-mini...
Re-running data_preprocessing node for 2020 - 8 with gpt-4o-mini...
Re-running data_preprocessing node for 2021 - 28 with gpt-4o-mini...
Re-running data_preprocessing node for 2021 - 56 with gpt-4o-mini...
Re-running data_preprocessing node for 2023 - 58 with gpt-4o-mini...
Re-running data_preprocessing node for 2024 - 99 with gpt-4o-mini...

----------------------------------------------------------------------------------------------------
RE-TEST RESULTS (compact summary)
----------------------------------------------------------------------------------------------------
 Paper ID       Model        GT_Data Cleaning New_Agent_Data Cleaning Match_Data Cleaning GT_Data Reduction New_Agent_Data Reduction Match_Data Reduction  GT_Data Transformation New_Agent_Data Transformation Match_Data Transformation Match_All_3
 2018 - 8 gpt-4o-mini            Not Reported            Not Reported               MATCH      Not Repor

**Iteration 5** After changing prompt according to 2021 - 28, 'Data Transformation'

In [16]:
# Batch run (quiet reasoning, but show assigned labels per paper)
retest_df = run_data_preprocessing_iteration(
    papers_to_rerun,
    llm_model="gpt-4o-mini",
    verbose=True,
    include_reasoning=True,
    show_labels_in_logs=False,
    show_reasoning_in_logs=False,
    log_bibliography=False,
    show_raw_label_output=True,
 )

Re-running data_preprocessing node for 2018 - 8 with gpt-4o-mini...
Re-running data_preprocessing node for 2019 - 33 with gpt-4o-mini...
Re-running data_preprocessing node for 2020 - 8 with gpt-4o-mini...
Re-running data_preprocessing node for 2021 - 28 with gpt-4o-mini...
Re-running data_preprocessing node for 2021 - 56 with gpt-4o-mini...
Re-running data_preprocessing node for 2023 - 58 with gpt-4o-mini...
Re-running data_preprocessing node for 2024 - 99 with gpt-4o-mini...

----------------------------------------------------------------------------------------------------
RE-TEST RESULTS (compact summary)
----------------------------------------------------------------------------------------------------
 Paper ID       Model        GT_Data Cleaning New_Agent_Data Cleaning Match_Data Cleaning GT_Data Reduction New_Agent_Data Reduction Match_Data Reduction  GT_Data Transformation New_Agent_Data Transformation Match_Data Transformation Match_All_3
 2018 - 8 gpt-4o-mini            Not

**Iteration 6** After working on 2023 - 58, adding rules such as Photo Labelling using Google Vision to be a Transparetly described Data Transformation. 

In [13]:
# Batch run (quiet reasoning, but show assigned labels per paper)
retest_df = run_data_preprocessing_iteration(
    papers_to_rerun,
    llm_model="gpt-4o-mini",
    verbose=True,
    include_reasoning=True,
    show_labels_in_logs=False,
    show_reasoning_in_logs=False,
    log_bibliography=False,
    show_raw_label_output=True,
 )

Re-running data_preprocessing node for 2018 - 8 with gpt-4o-mini...
Re-running data_preprocessing node for 2019 - 33 with gpt-4o-mini...
Re-running data_preprocessing node for 2020 - 8 with gpt-4o-mini...
Re-running data_preprocessing node for 2021 - 28 with gpt-4o-mini...
Re-running data_preprocessing node for 2021 - 56 with gpt-4o-mini...
Re-running data_preprocessing node for 2023 - 58 with gpt-4o-mini...
Re-running data_preprocessing node for 2024 - 99 with gpt-4o-mini...

----------------------------------------------------------------------------------------------------
RE-TEST RESULTS (compact summary)
----------------------------------------------------------------------------------------------------
 Paper ID       Model        GT_Data Cleaning New_Agent_Data Cleaning Match_Data Cleaning       GT_Data Reduction New_Agent_Data Reduction Match_Data Reduction  GT_Data Transformation New_Agent_Data Transformation Match_Data Transformation Match_All_3
 2018 - 8 gpt-4o-mini         

In [14]:
# Batch run (quiet reasoning, but show assigned labels per paper)
retest_df = run_data_preprocessing_iteration(
    papers_to_rerun,
    llm_model="gpt-4o-mini",
    verbose=True,
    include_reasoning=True,
    show_labels_in_logs=False,
    show_reasoning_in_logs=False,
    log_bibliography=False,
    show_raw_label_output=True,
 )

Re-running data_preprocessing node for 2018 - 8 with gpt-4o-mini...
Re-running data_preprocessing node for 2019 - 33 with gpt-4o-mini...
Re-running data_preprocessing node for 2020 - 8 with gpt-4o-mini...
Re-running data_preprocessing node for 2021 - 28 with gpt-4o-mini...
Re-running data_preprocessing node for 2021 - 56 with gpt-4o-mini...
Re-running data_preprocessing node for 2023 - 58 with gpt-4o-mini...
Re-running data_preprocessing node for 2024 - 99 with gpt-4o-mini...

----------------------------------------------------------------------------------------------------
RE-TEST RESULTS (compact summary)
----------------------------------------------------------------------------------------------------
 Paper ID       Model        GT_Data Cleaning New_Agent_Data Cleaning Match_Data Cleaning       GT_Data Reduction New_Agent_Data Reduction Match_Data Reduction  GT_Data Transformation New_Agent_Data Transformation Match_Data Transformation Match_All_3
 2018 - 8 gpt-4o-mini         

In [35]:
# Batch run (quiet reasoning, but show assigned labels per paper)
retest_df = run_data_preprocessing_iteration(
    papers_to_rerun,
    llm_model="gpt-4o-mini",
    verbose=True,
    include_reasoning=True,
    show_labels_in_logs=False,
    show_reasoning_in_logs=False,
    log_bibliography=False,
    show_raw_label_output=True,
 )

Re-running data_preprocessing node for 2018 - 8 with gpt-4o-mini...
Re-running data_preprocessing node for 2019 - 33 with gpt-4o-mini...
Re-running data_preprocessing node for 2020 - 8 with gpt-4o-mini...
Re-running data_preprocessing node for 2021 - 28 with gpt-4o-mini...
Re-running data_preprocessing node for 2021 - 56 with gpt-4o-mini...
Re-running data_preprocessing node for 2023 - 58 with gpt-4o-mini...
Re-running data_preprocessing node for 2024 - 99 with gpt-4o-mini...

----------------------------------------------------------------------------------------------------
RE-TEST RESULTS (compact summary)
----------------------------------------------------------------------------------------------------
 Paper ID       Model        GT_Data Cleaning New_Agent_Data Cleaning Match_Data Cleaning       GT_Data Reduction New_Agent_Data Reduction Match_Data Reduction  GT_Data Transformation New_Agent_Data Transformation Match_Data Transformation Match_All_3
 2018 - 8 gpt-4o-mini         

**Iteration** Last.

In [11]:
# Batch run (quiet reasoning, but show assigned labels per paper)
retest_df = run_data_preprocessing_iteration(
    papers_to_rerun,
    llm_model="gpt-4o-mini",
    verbose=True,
    include_reasoning=True,
    show_labels_in_logs=False,
    show_reasoning_in_logs=False,
    log_bibliography=False,
    show_raw_label_output=True,
 )

Re-running data_preprocessing node for 2018 - 8 with gpt-4o-mini...
Re-running data_preprocessing node for 2019 - 33 with gpt-4o-mini...
Re-running data_preprocessing node for 2020 - 8 with gpt-4o-mini...
Re-running data_preprocessing node for 2021 - 28 with gpt-4o-mini...
Re-running data_preprocessing node for 2021 - 56 with gpt-4o-mini...
Re-running data_preprocessing node for 2023 - 58 with gpt-4o-mini...
Re-running data_preprocessing node for 2024 - 99 with gpt-4o-mini...

----------------------------------------------------------------------------------------------------
RE-TEST RESULTS (compact summary)
----------------------------------------------------------------------------------------------------
 Paper ID       Model        GT_Data Cleaning New_Agent_Data Cleaning Match_Data Cleaning GT_Data Reduction New_Agent_Data Reduction Match_Data Reduction  GT_Data Transformation New_Agent_Data Transformation Match_Data Transformation Match_All_3
 2018 - 8 gpt-4o-mini            Not

### Post-finalization reasoning-cum-debugging view
Use this after batch run to inspect only the papers you care about (for example only mismatches).

In [24]:
# 1) Reasoning for only mismatched papers
watch_reasoning_post_finalization(
    retest_df,
    paper_ids= ["2024 - 99"],
    only_mismatches=True,
    show_raw_label_output=False,
    show_bibliography=True,
 )

# 2) Or reasoning for selected papers
# watch_reasoning_post_finalization(
#     retest_df,
#     paper_ids=["2021 - 28", "2020 - 8"],
#     show_raw_label_output=False,
#     show_bibliography=True,
# )

Showing reasoning for 1 paper(s).

PAPER: 2023 - 58 | MODEL: gpt-4o-mini

Data Cleaning
  GT    : Mentioned
  Agent : Not Reported
  Match : MISMATCH
  Justification:
    Status: Not Reported. No evidence of missing value handling found in text. Therefore classified as Not
    Reported per DSRP Data Cleaning definition.
  Bibliography:
    (no bibliography)

Data Reduction
  GT    : Not Reported
  Agent : Not Reported
  Match : MATCH
  Justification:
    Status: Not Reported. No evidence of data reduction techniques found in text. Thus classified as Not
    Reported according to DSRP definitions.
  Bibliography:
    (no bibliography)

Data Transformation
  GT    : Mentioned
  Agent : Not Reported
  Match : MISMATCH
  Justification:
    Status: Not Reported. No evidence of data transformation procedures identified in the text. Therefore
    classified as Not Reported per DSRP Data Transformation definition.
  Bibliography:
    (no bibliography)

Overall all-3-dimensions match: MISMATCH


#### **Re-run on Single ID**

In [10]:
# Single-paper readable debug output (multi-line evidence)
run_data_preprocessing_iteration(
    ["2024 - 99"],
    llm_model="gpt-4o-mini",
    verbose=True,
    include_reasoning=True,
    show_reasoning_in_logs=True,
    log_bibliography=True,
    show_raw_label_output=False,
 )

Re-running data_preprocessing node for 2024 - 99 with gpt-4o-mini...
  Assigned labels -> Data Cleaning: Not Reported | Data Reduction: Not Reported | Data Transformation: Not Reported
  Match_All_3: MATCH

PAPER: 2024 - 99 | MODEL: gpt-4o-mini

Data Cleaning
  GT    : Not Reported
  Agent : Not Reported
  Match : MATCH
  Justification:
    Status: Not Reported. No evidence of data cleaning operations found in the text. Therefore classified as
    Not Reported per DSRP Data Cleaning definition.
  Bibliography:
    (no bibliography)

Data Reduction
  GT    : Not Reported
  Agent : Not Reported
  Match : MATCH
  Justification:
    Status: Not Reported. No evidence of data reduction procedures found in the text. Therefore classified as
    Not Reported per DSRP Data Reduction definition.
  Bibliography:
    (no bibliography)

Data Transformation
  GT    : Not Reported
  Agent : Not Reported
  Match : MATCH
  Justification:
    Status: Not Reported. No evidence of data transformation activ

,Paper ID,Model,GT_Data Cleaning,New_Agent_Data Cleaning,Match_Data Cleaning,GT_Data Reduction,New_Agent_Data Reduction,Match_Data Reduction,GT_Data Transformation,New_Agent_Data Transformation,...,confidence,validated_bibliography_count,validated_bibliography_lines,Justification_Data Cleaning,Raw_Data Cleaning_obj,Justification_Data Reduction,Raw_Data Reduction_obj,Justification_Data Transformation,Raw_Data Transformation_obj,Match_All_3
0,2024 - 99,gpt-4o-mini,Not Reported,Not Reported,MATCH,Not Reported,Not Reported,MATCH,Not Reported,Not Reported,...,1.0,0,[(no bibliography)],Status: Not Reported. No evidence of data clea...,"{\n ""status"": ""Not Reported"",\n ""justificati...",Status: Not Reported. No evidence of data redu...,"{\n ""status"": ""Not Reported"",\n ""justificati...",Status: Not Reported. No evidence of data tran...,"{\n ""status"": ""Not Reported"",\n ""justificati...",MATCH
